In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

import torch_geometric
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, global_mean_pool
from torch_geometric.utils import negative_sampling

from utils.data_pool_gt import load_ground_truth_data
from utils.constants import get_classlist_and_classdict
from utils import constants

from typing import List
#mung 
from mung.io import parse_node_classes, read_nodes_from_file
from mung.graph import NotationGraph
from mung.grammar import DependencyGrammar

import cv2

In [2]:
# restricted classes
node_classes_path = "data/MUSCIMA++/v2.0/specifications/mff-muscima-mlclasses-annot.xml"
CLASS_LIST_20 = constants.CLASS_LIST_20
CLASS_DICT_ALL = {node_class.name : node_class.class_id for node_class in parse_node_classes(node_classes_path)}


## Restrict imbalance frequency

In [3]:
CLASS_ESSN = constants.CLASS_LIST_ESSN

In [4]:
class_dict_essn = {node_class.name : node_class.class_id for node_class in parse_node_classes(node_classes_path) if node_class.name in CLASS_ESSN}
class_dict_essn

{'noteheadFull': 0,
 'stem': 2,
 'beam': 7,
 'augmentationDot': 8,
 'accidentalSharp': 9,
 'accidentalFlat': 10,
 'accidentalNatural': 11,
 'accidentalDoubleSharp': 12,
 'accidentalDoubleFlat': 13,
 'restWhole': 14,
 'restHalf': 15,
 'restQuarter': 16,
 'rest8th': 17,
 'rest16th': 18,
 'multiMeasureRest': 21,
 'repeat1Bar': 22,
 'legerLine': 23,
 'graceNoteAcciaccatura': 24,
 'noteheadFullSmall': 25,
 'brace': 36,
 'staffGrouping': 37,
 'barline': 38,
 'barlineHeavy': 39,
 'measureSeparator': 40,
 'repeat': 41,
 'repeatDot': 42,
 'articulationStaccato': 46,
 'articulationTenuto': 48,
 'articulationAccent': 49,
 'slur': 52,
 'tie': 53,
 'dynamicCrescendoHairpin': 54,
 'dynamicDiminuendoHairpin': 55,
 'ornament': 56,
 'wiggleTrill': 57,
 'ornamentTrill': 58,
 'arpeggio': 59,
 'glissando': 60,
 'tupleBracket': 65,
 'tuple': 66,
 'gClef': 67,
 'fClef': 68,
 'cClef': 69,
 'keySignature': 71,
 'timeSignature': 72,
 'dynamicsText': 77,
 'tempoText': 78,
 'otherText': 82,
 'numeral0': 135,
 'n

In [5]:
# Data paths (same as MLP)
gt_annotations_root = 'data/MUSCIMA++/v2.0/data/annotations'
images_root = 'data/MUSCIMA++/datasets_r_staff/images'
split_file = 'splits/mob_split.yaml'

## Load NotationGraph

In [6]:
def __load_mung(filename: str) -> NotationGraph:
    mungos = read_nodes_from_file(os.path.join(gt_annotations_root, filename))
    mung = NotationGraph(mungos)
    return mung

In [7]:
# def __load_mung(filename: str, included_classes: List[str]) -> NotationGraph:
#     mungos = read_nodes_from_file(os.path.join(gt_annotations_root, filename))
#     mung = NotationGraph(mungos)
#     objects_to_include = [m for m in mungos if m.class_name in included_classes]
#     for m in mungos:
#         if m not in objects_to_include:
#             mung.remove_vertex(m.id)
#     return mung

In [8]:
# Load dataset
import yaml
with open("splits/mob_split.yaml", "rb") as f:
    split = yaml.load(f, Loader = yaml.FullLoader)
    
train_files = split['train']
valid_files = split['valid']
test_files = split['test']

In [9]:
images_train = [path + '.png' for path in train_files]
images_valid = [path + '.png' for path in valid_files]
images_test = [path + '.png' for path in test_files]

In [10]:
# all path
parent = "data/MUSCIMA++/v2.0/data/annotations"
paths = [p for p in os.listdir(parent) if p.endswith('.xml')]
image_paths = [path.replace('.xml','.png') for path in paths]


In [11]:

# def mung_to_pyg(mung, image_path, class_dict, images_root, normalize_box: bool = True) -> Data:
  
#     image_full_path = os.path.join(images_root, image_path)
#     image = cv2.imread(image_full_path)
#     if image is None:
#         raise FileNotFoundError(f"Could not read image: {image_full_path}")

#     H, W = image.shape[:2]  #height,width
#     x_list = []
#     #--- Build node features ---
#     for node in mung.vertices:
#         bbox = np.array(node.bounding_box, dtype=np.float32)
#         x_min, y_min, x_max, y_max = bbox
#         if normalize_box:
#             bbox = np.array([
#                 x_min / W,
#                 y_min / H,
#                 x_max / W,
#                 y_max / H
#             ], dtype=np.float32)

#         cls_id = class_dict.get(node.class_name, -1)
#         if cls_id == -1:
#             raise KeyError(f"Class name '{node.class_name}' not found in class_dict.")

#         #Concatenate [x_min, y_min, x_max, y_max, class_id]
#         node_feat = np.concatenate([bbox, [cls_id]])
#         x_list.append(node_feat)

#     x = torch.tensor(x_list, dtype=torch.float32)

#     # --- Build edges ---
#     edge_index = []
#     for node in mung.vertices:
#         for target_id in node.outlinks:
#             edge_index.append([node.id, target_id])
#     if len(edge_index) == 0:
#         # Handle empty edge case gracefully
#         edge_index = torch.empty((2, 0), dtype=torch.long)
#     else:
#         edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
#     # --- Assemble PyG Data object ---
#     data = Data(x=x, edge_index=edge_index)
#     return data


In [12]:

def mung_to_pyg(mung, image_path, class_dict, images_root, normalize_box: bool = True) -> Data:
    """
    Convert a MUSCIMA++ NotationGraph into a PyTorch Geometric Data graph.
    """
    # --- Load image safely ---
    image_full_path = os.path.join(images_root, image_path)
    image = cv2.imread(image_full_path)
    H, W = image.shape[:2]
    vertices = mung.vertices
    # --- Create mapping from node.id → contiguous index ---
    id_to_idx = {node.id: i for i, node in enumerate(vertices)}
    # --- Build node features ---
    x_list = []
    for node in vertices:
        bbox = np.array(node.bounding_box, dtype=np.float32)
        x_min, y_min, x_max, y_max = bbox
        
        if normalize_box:
            bbox = np.array([
                x_min / W,
                y_min / H,
                x_max / W,
                y_max / H
            ], dtype=np.float32)

        cls_id = class_dict.get(node.class_name, -1)
        if cls_id == -1:
            raise KeyError(f"Class '{node.class_name}' not found in class_dict")

        node_feat = np.concatenate([bbox, [cls_id]])
        x_list.append(node_feat)

    x = torch.tensor(x_list, dtype=torch.float32)

    # --- Build edges with reindexed node IDs ---
    edge_index = []
    for node in vertices:
        for target_id in node.outlinks:
            if target_id in id_to_idx:
                edge_index.append([id_to_idx[node.id], id_to_idx[target_id]])
            # else: skip edges to non-existent nodes (robustness)

    if len(edge_index) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    data = Data(x=x, edge_index=edge_index)
    return data


In [13]:
mungs_train = [ __load_mung(path+'.xml') for path in train_files]
mungs_valid = [ __load_mung(path+'.xml') for path in valid_files]
mungs_test = [ __load_mung(path+'.xml') for path in test_files]

## DataLoader

In [14]:
from torch_geometric.loader import DataLoader

graphs_train = [mung_to_pyg(mung = m, image_path= image_path,class_dict = CLASS_DICT_ALL,images_root=images_root , normalize_box= True) for m , image_path in zip(mungs_train,images_train)]

/var/folders/_9/m2ryk39s2slc67mc6y5xdxcm0000gn/T/ipykernel_51827/2525780990.py:33: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  x = torch.tensor(x_list, dtype=torch.float32)


In [15]:
graphs_valid = [mung_to_pyg(mung = m, image_path= image_path,class_dict = CLASS_DICT_ALL,images_root=images_root , normalize_box= True) for m , image_path in zip(mungs_valid,images_valid)]

In [16]:
graphs_test = [mung_to_pyg(mung = m, image_path= image_path,class_dict = CLASS_DICT_ALL,images_root=images_root , normalize_box= True) for m , image_path in zip(mungs_test,images_test)]

In [17]:
# from sklearn.model_selection import train_test_split

# train_graphs, val_graphs = train_test_split(graphs, test_size=0.2, random_state=42)

train_loader = DataLoader(graphs_train, batch_size=1, shuffle=True)
val_loader   = DataLoader(graphs_valid,   batch_size=1, shuffle=False)
test_loader = DataLoader(graphs_test,batch_size= 1, shuffle=True )

In [18]:
mungs_valid = [ __load_mung(path+'.xml') for path in valid_files]
graphs_valid = [mung_to_pyg(mung = m, image_path= image_path,class_dict = CLASS_DICT_ALL,images_root=images_root , normalize_box= True) for m , image_path in zip(mungs_valid,images_valid)]
val_loader   = DataLoader(graphs_valid,   batch_size=1, shuffle=False)



def mung_to_pyg(mung, image_path, class_dict, images_root, normalize_box: bool = True) -> Data:
    """
    Convert a MUSCIMA++ NotationGraph into a PyTorch Geometric Data graph.
    """
    # --- Load image safely ---
    image_full_path = os.path.join(images_root, image_path)
    image = cv2.imread(image_full_path)
    H, W = image.shape[:2]
    vertices = mung.vertices
    # --- Create mapping from node.id → contiguous index ---
    id_to_idx = {node.id: i for i, node in enumerate(vertices)}
    # --- Build node features ---
    x_list = []
    for node in vertices:
        bbox = np.array(node.bounding_box, dtype=np.float32)
        x_min, y_min, x_max, y_max = bbox
        
        if normalize_box:
            bbox = np.array([
                x_min / W,
                y_min / H,
                x_max / W,
                y_max / H
            ], dtype=np.float32)

        cls_id = class_dict.get(node.class_name, -1)
        if cls_id == -1:
            raise KeyError(f"Class '{node.class_name}' not found in class_dict")

        node_feat = np.concatenate([bbox, [cls_id]])
        x_list.append(node_feat)

    x = torch.tensor(x_list, dtype=torch.float32)

    # --- Build edges with reindexed node IDs ---
    edge_index = []
    for node in vertices:
        for target_id in node.outlinks:
            if target_id in id_to_idx:
                edge_index.append([id_to_idx[node.id], id_to_idx[target_id]])
            # else: skip edges to non-existent nodes (robustness)

    if len(edge_index) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    data = Data(x=x, edge_index=edge_index)
    return data


## GNN encoder

In [19]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GraphNorm


In [20]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GraphNorm

class GNNEncoder(torch.nn.Module):
    """
    Basic 2-layer Graph Convolutional Network (GCN) encoder.
    Encodes node features (bbox + class) into latent embeddings.
    """
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.norm1 = GraphNorm(hidden_dim)
        self.conv2 = GCNConv(hidden_dim, out_dim)
        self.norm2 = GraphNorm(out_dim)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.norm1(x)
        x = F.relu(self.conv2(x, edge_index))
        x = self.norm2(x)
        return x  


In [21]:
class LinkPredictor(torch.nn.Module):
    """
    Predicts whether an edge exists between two nodes based on their embeddings.
    """
    def __init__(self, embed_dim):
        super().__init__()
        self.fc = torch.nn.Sequential(
            torch.nn.Linear(embed_dim * 2, embed_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(embed_dim, 1),
            torch.nn.Sigmoid()
        )

    def forward(self, src, dst):
        h = torch.cat([src, dst], dim=-1)
        return self.fc(h)


In [22]:
from torch_geometric.utils import negative_sampling

def get_edge_samples(data):
    pos_edge_index = data.edge_index
    neg_edge_index = negative_sampling(
        edge_index=pos_edge_index,
        num_nodes=data.num_nodes,
        num_neg_samples=pos_edge_index.size(1)
    )
    return pos_edge_index, neg_edge_index


In [23]:
for data in train_loader:
    print(data)
    break

DataBatch(x=[547, 5], edge_index=[2, 722], batch=[547], ptr=[2])


In [24]:
from pdb import set_trace

In [25]:
def train_epoch(encoder, predictor, loader, optimizer, device):
    encoder.train()
    predictor.train()
    total_loss = 0

    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()

        z = encoder(data.x, data.edge_index)  # node embeddings

        pos_edge_index, neg_edge_index = get_edge_samples(data)
   

        # Positive samples
        pos_out = predictor(z[pos_edge_index[0]], z[pos_edge_index[1]])
        pos_label = torch.ones(pos_out.size(0), 1, device=device)

        # Negative samples
        neg_out = predictor(z[neg_edge_index[0]], z[neg_edge_index[1]])
        neg_label = torch.zeros(neg_out.size(0), 1, device=device)

        # Concatenate and compute loss
        out = torch.cat([pos_out, neg_out], dim=0)
        label = torch.cat([pos_label, neg_label], dim=0)
        loss = F.binary_cross_entropy(out, label)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        

    return total_loss / len(loader)


In [26]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

in_dim = 5      
hidden_dim = 64
out_dim = 64

encoder = GNNEncoder(in_dim, hidden_dim, out_dim).to(device)
predictor = LinkPredictor(out_dim).to(device)

optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(predictor.parameters()), lr=1e-3
)

for epoch in range(1, 21):
    loss = train_epoch(encoder, predictor, train_loader, optimizer, device)
    print(f"Epoch {epoch:02d} | Loss: {loss:.4f}")


Epoch 01 | Loss: 0.4046
Epoch 02 | Loss: 0.2313
Epoch 03 | Loss: 0.1669
Epoch 04 | Loss: 0.1259
Epoch 05 | Loss: 0.0991
Epoch 06 | Loss: 0.0859
Epoch 07 | Loss: 0.0779
Epoch 08 | Loss: 0.0704
Epoch 09 | Loss: 0.0630
Epoch 10 | Loss: 0.0583
Epoch 11 | Loss: 0.0562
Epoch 12 | Loss: 0.0549
Epoch 13 | Loss: 0.0517
Epoch 14 | Loss: 0.0472
Epoch 15 | Loss: 0.0540
Epoch 16 | Loss: 0.0473
Epoch 17 | Loss: 0.0413
Epoch 18 | Loss: 0.0401
Epoch 19 | Loss: 0.0362
Epoch 20 | Loss: 0.0415


In [55]:
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
from torch_geometric.utils import negative_sampling

def evaluate(encoder, predictor, loader, device, fixed_neg=False):
    """
    Evaluate GNN link prediction performance using ROC-AUC.

    Parameters
    ----------
    encoder : nn.Module
        GNN encoder that computes node embeddings.
    predictor : nn.Module
        MLP or GNN-based link predictor.
    loader : DataLoader
        PyG DataLoader with graphs to evaluate.
    device : torch.device
        CUDA or CPU device.
    fixed_neg : bool, optional
        If True, reuses the same negative samples for consistency across epochs.

    Returns
    -------
    auc : float
        ROC-AUC score (0.5 = random, 1.0 = perfect).
    """
    encoder.eval()
    predictor.eval()

    y_true, y_pred = [], []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)

            z = encoder(data.x, data.edge_index)

            pos_edge_index = data.edge_index
            # Reuse or create negative samples
            if fixed_neg and hasattr(data, "neg_edge_index"):
                neg_edge_index = data.neg_edge_index
            else:
                neg_edge_index = negative_sampling(
                    edge_index=pos_edge_index,
                    num_nodes=data.num_nodes,
                    num_neg_samples=pos_edge_index.size(1)
                )
                if fixed_neg:
                    data.neg_edge_index = neg_edge_index

            pos_out = predictor(z[pos_edge_index[0]], z[pos_edge_index[1]])
            neg_out = predictor(z[neg_edge_index[0]], z[neg_edge_index[1]])

            pos_scores = pos_out.detach().cpu().squeeze().numpy()
            neg_scores = neg_out.detach().cpu().squeeze().numpy()

            y_true.extend([1] * len(pos_scores))
            y_true.extend([0] * len(neg_scores))
            y_pred.extend(pos_scores.tolist())
            y_pred.extend(neg_scores.tolist())

    # Guard against degenerate cases
    if len(set(y_true)) < 2:
        print("⚠️ Only one class present in y_true, returning 0.5 baseline.")
        return 0.5
 

    auc = roc_auc_score(y_true, y_pred)
    return auc


In [56]:
auc = evaluate(encoder, predictor, val_loader, device)
print(f"Validation ROC-AUC: {auc:.4f}")


Validation ROC-AUC: 0.9986


In [57]:
for epoch in range(1, 21):
    loss = train_epoch(encoder, predictor, train_loader, optimizer, device)
    auc = evaluate(encoder, predictor, val_loader, device, fixed_neg=True)
    print(f"Epoch {epoch:02d} | Train Loss: {loss:.4f} | Val AUC: {auc:.4f}")


Epoch 01 | Train Loss: 0.0270 | Val AUC: 0.9983
Epoch 02 | Train Loss: 0.0255 | Val AUC: 0.9987
Epoch 03 | Train Loss: 0.0297 | Val AUC: 0.9973
Epoch 04 | Train Loss: 0.0333 | Val AUC: 0.9984
Epoch 05 | Train Loss: 0.0252 | Val AUC: 0.9985
Epoch 06 | Train Loss: 0.0233 | Val AUC: 0.9987
Epoch 07 | Train Loss: 0.0221 | Val AUC: 0.9987
Epoch 08 | Train Loss: 0.0237 | Val AUC: 0.9984
Epoch 09 | Train Loss: 0.0213 | Val AUC: 0.9988
Epoch 10 | Train Loss: 0.0299 | Val AUC: 0.9983
Epoch 11 | Train Loss: 0.0231 | Val AUC: 0.9988
Epoch 12 | Train Loss: 0.0221 | Val AUC: 0.9988
Epoch 13 | Train Loss: 0.0208 | Val AUC: 0.9988
Epoch 14 | Train Loss: 0.0218 | Val AUC: 0.9986
Epoch 15 | Train Loss: 0.0229 | Val AUC: 0.9987
Epoch 16 | Train Loss: 0.0202 | Val AUC: 0.9990
Epoch 17 | Train Loss: 0.0216 | Val AUC: 0.9987
Epoch 18 | Train Loss: 0.0226 | Val AUC: 0.9987
Epoch 19 | Train Loss: 0.0214 | Val AUC: 0.9990
Epoch 20 | Train Loss: 0.0205 | Val AUC: 0.9984


## Test

In [ ]:
# from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
# from torch_geometric.utils import negative_sampling
# import numpy as np

# def evaluate(encoder, predictor, loader, device, fixed_neg=False, threshold=0.5):
#     """
#     Evaluate GNN link prediction performance using ROC-AUC and F1-score.

#     Parameters
#     ----------
#     encoder : nn.Module
#         GNN encoder that computes node embeddings.
#     predictor : nn.Module
#         MLP or GNN-based link predictor.
#     loader : DataLoader
#         PyG DataLoader with graphs to evaluate.
#     device : torch.device
#         CUDA or CPU device.
#     fixed_neg : bool, optional
#         If True, reuses the same negative samples for consistency across epochs.
#     threshold : float, optional
#         Decision threshold for computing binary predictions (default=0.5).

#     Returns
#     -------
#     metrics : dict
#         Dictionary with:
#             - 'roc_auc' : float
#             - 'precision' : float
#             - 'recall' : float
#             - 'f1' : float
#     """
#     encoder.eval()
#     predictor.eval()

#     y_true, y_pred = [], []

#     with torch.no_grad():
#         for data in loader:
#             data = data.to(device)

#             z = encoder(data.x, data.edge_index)

#             pos_edge_index = data.edge_index
#             # Reuse or create negative samples
#             if fixed_neg and hasattr(data, "neg_edge_index"):
#                 neg_edge_index = data.neg_edge_index
#             else:
#                 neg_edge_index = negative_sampling(
#                     edge_index=pos_edge_index,
#                     num_nodes=data.num_nodes,
#                     num_neg_samples=pos_edge_index.size(1)
#                 )
#                 if fixed_neg:
#                     data.neg_edge_index = neg_edge_index

#             pos_out = predictor(z[pos_edge_index[0]], z[pos_edge_index[1]])
#             neg_out = predictor(z[neg_edge_index[0]], z[neg_edge_index[1]])

#             pos_scores = pos_out.detach().cpu().squeeze().numpy()
#             neg_scores = neg_out.detach().cpu().squeeze().numpy()

#             y_true.extend([1] * len(pos_scores))
#             y_true.extend([0] * len(neg_scores))
#             y_pred.extend(pos_scores.tolist())
#             y_pred.extend(neg_scores.tolist())

#     # Guard against degenerate cases
#     if len(set(y_true)) < 2:
#         print("⚠️ Only one class present in y_true; returning baseline scores.")
#         return {'roc_auc': 0.5, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}

#     # Convert to numpy arrays for metric computation
#     y_true = np.array(y_true)
#     y_pred = np.array(y_pred)

#     # ROC-AUC (continuous scores)
#     roc_auc = roc_auc_score(y_true, y_pred)

#     # Convert probabilities to binary decisions
#     y_pred_binary = (y_pred >= threshold).astype(int)

#     # Compute precision, recall, F1
#     precision, recall, f1, _ = precision_recall_fscore_support(
#         y_true, y_pred_binary, average='binary', zero_division=0
#     )

#     return {
#         'roc_auc': roc_auc,
#         'precision': precision,
#         'recall': recall,
#         'f1': f1
#     }


In [54]:
from sklearn.metrics import roc_auc_score, average_precision_score
from torch_geometric.utils import negative_sampling

@torch.no_grad()
def evaluate_auc(encoder, predictor, loader, device):
    """
    Evaluate link prediction performance (AUC + AP) like MovieLens tutorial.
    Works per-page (per-graph) in OMR setting.
    """
    encoder.eval()
    predictor.eval()
    aucs, aps = [], []

    for data in loader:
        data = data.to(device)

        # Encode with known structure (only for validation)
        z = encoder(data.x, data.edge_index)

        # Positive edges = ground truth
        pos_edge_index = data.edge_index

        # Negative edges = random pairs
        neg_edge_index = negative_sampling(
            edge_index=pos_edge_index,
            num_nodes=data.num_nodes,
            num_neg_samples=pos_edge_index.size(1)
        )

        # Score both sets
        pos_pred = predictor(z[pos_edge_index[0]], z[pos_edge_index[1]]).squeeze()
        neg_pred = predictor(z[neg_edge_index[0]], z[neg_edge_index[1]]).squeeze()

        y_true = torch.cat([
            torch.ones(pos_pred.size(0), device=device),
            torch.zeros(neg_pred.size(0), device=device)
        ])
        y_pred = torch.cat([pos_pred, neg_pred])

        # Compute AUC and Average Precision
        auc = roc_auc_score(y_true.cpu(), y_pred.cpu())
        ap = average_precision_score(y_true.cpu(), y_pred.cpu())

        aucs.append(auc)
        aps.append(ap)

    return np.mean(aucs), np.mean(aps)
val_auc, val_ap = evaluate_auc(encoder, predictor, val_loader, device)
print(f"Validation AUC: {val_auc:.4f}, AP: {val_ap:.4f}")


Validation AUC: 0.9986, AP: 0.9981


## Save Model

In [58]:
torch.save(encoder.state_dict(), "outputs/gnn_1/encoder.pt")
torch.save(predictor.state_dict(), "outputs/gnn_1/predictor.pt")


## Load Model

In [60]:
encoder.load_state_dict(torch.load("outputs/gnn_1/encoder.pt", map_location=device))
predictor.load_state_dict(torch.load("outputs/gnn_1/predictor.pt", map_location=device))
encoder.eval()
predictor.eval()


LinkPredictor(
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
    (3): Sigmoid()
  )
)

## Test with new data

In [ ]:

    # # --- Load image safely ---
    # image_full_path = os.path.join(images_root, image_path)
    # image = cv2.imread(image_full_path)
    # H, W = image.shape[:2]
    # vertices = mung.vertices
    # # --- Create mapping from node.id → contiguous index ---
    # id_to_idx = {node.id: i for i, node in enumerate(vertices)}
    # # --- Build node features ---
    # x_list = []
    # for node in vertices:
    #     bbox = np.array(node.bounding_box, dtype=np.float32)
    #     x_min, y_min, x_max, y_max = bbox
        
    #     if normalize_box:
    #         bbox = np.array([
    #             x_min / W,
    #             y_min / H,
    #             x_max / W,
    #             y_max / H
    #         ], dtype=np.float32)


In [89]:
test_image_path = os.path.join(images_root,"CVC-MUSCIMA_W-01_N-10_D-ideal.png")

test_mung = __load_mung("CVC-MUSCIMA_W-01_N-10_D-ideal.png".replace('.png','.xml'))




In [ ]:
test_image = cv2.imread(test_image_path)
H,W = test_image.shape[:2]

In [84]:
x_list = []
for node in test_mung.vertices:
    x_min,y_min, x_max, y_max = node.bounding_box
    cls_id = CLASS_DICT_ALL.get(node.class_name,-1)
    if cls_id == -1:
            continue

    bbox = np.array([x_min/W, y_min/H, x_max/W, y_max/H], dtype=np.float32)
    node_feat = np.concatenate([bbox, [cls_id]])
    x_list.append(node_feat)

x = torch.tensor(x_list, dtype=torch.float32)
data = Data(x=x, edge_index=torch.empty((2, 0), dtype=torch.long))
data

Data(x=[807, 5], edge_index=[2, 0])

## Predict without knowledge of edges

In [85]:
def predict_edges_from_features(encoder, predictor, data, device, distance_threshold=None, prob_threshold=0.5):
    """
    Predict edges in a graph without known connections, e.g. YOLO-detected symbols.
    """
    data = data.to(device)
    data.edge_index = torch.empty((2, 0), dtype=torch.long).to(device)

    with torch.no_grad():
        z = encoder(data.x, data.edge_index)

        num_nodes = data.num_nodes
        pairs = torch.combinations(torch.arange(num_nodes, device=device), r=2)
       

        # Optional geometric filtering
        if distance_threshold is not None:
            coords = data.x[:, :4].cpu().numpy()
            centers = np.column_stack(((coords[:, 0] + coords[:, 2]) / 2,
                                       (coords[:, 1] + coords[:, 3]) / 2))
            valid_pairs = []
            for i, j in pairs.cpu().numpy():
                if np.linalg.norm(centers[i] - centers[j]) < distance_threshold:
                    valid_pairs.append((i, j))
            pairs = torch.tensor(valid_pairs, dtype=torch.long, device=device)

        scores = predictor(z[pairs[:, 0]], z[pairs[:, 1]]).squeeze()
        predicted_edges = pairs[scores > prob_threshold]

    return predicted_edges, scores


In [86]:
predicted_edges, scores = predict_edges_from_features(
    encoder, predictor, data, device,
    distance_threshold=100,  # adjust based on image scale
    prob_threshold=0.5
)
print(f"Predicted {len(predicted_edges)} edges")


Predicted 3169 edges


## Visualize

In [91]:
import cv2

def visualize_edges(image_path, data, predicted_edges, save_path="predicted_edges.png"):
    image = cv2.imread(image_path)
    H, W = image.shape[:2]

    coords = data.x[:, :4].cpu().numpy()
    centers = np.column_stack(((coords[:, 0] + coords[:, 2]) / 2 * W,
                               (coords[:, 1] + coords[:, 3]) / 2 * H))

    for (i, j) in predicted_edges.cpu().numpy():
        p1, p2 = tuple(centers[i].astype(int)), tuple(centers[j].astype(int))
        cv2.line(image, p1, p2, (0, 255, 0), 1)

    cv2.imwrite(save_path, image)
    print(f"Saved visualization to {save_path}")


In [92]:
visualize_edges(test_image_path, data, predicted_edges)


Saved visualization to predicted_edges.png


In [ ]:
# for data in test_loader:
#     predicted_edges, scores = predict_edges_from_features(
#         encoder, predictor, data, device,
#         distance_threshold=100, prob_threshold=0.5
#     )
#     print(f"Predicted {len(predicted_edges)} edges for this page.")


Predicted 1847 edges for this page.
Predicted 3405 edges for this page.
Predicted 1673 edges for this page.
Predicted 4814 edges for this page.
Predicted 2940 edges for this page.
Predicted 3508 edges for this page.
Predicted 4748 edges for this page.
Predicted 1847 edges for this page.
Predicted 6982 edges for this page.
Predicted 2354 edges for this page.
Predicted 4684 edges for this page.
Predicted 2500 edges for this page.
Predicted 3445 edges for this page.
Predicted 3142 edges for this page.
Predicted 4012 edges for this page.
Predicted 3325 edges for this page.
Predicted 3287 edges for this page.
Predicted 3696 edges for this page.
Predicted 3148 edges for this page.
Predicted 1935 edges for this page.
Predicted 2341 edges for this page.
Predicted 2814 edges for this page.
Predicted 4044 edges for this page.
Predicted 6530 edges for this page.
Predicted 3969 edges for this page.
Predicted 1443 edges for this page.
Predicted 1648 edges for this page.
Predicted 6928 edges for thi

In [ ]:
# import numpy as np

# page_metrics = []

# for data in test_loader:
#     true_edge_index = data.edge_index
#     pred_edges, scores = predict_edges_from_features(
#         encoder, predictor, data, device,
#         distance_threshold=50, prob_threshold=0.3
#     )

#     gt_edges = set(map(tuple, true_edge_index.t().cpu().numpy()))
#     pred_edges_set = set(map(tuple, pred_edges.cpu().numpy()))

#     TP = len(gt_edges & pred_edges_set)
#     FP = len(pred_edges_set - gt_edges)
#     FN = len(gt_edges - pred_edges_set)

#     precision = TP / (TP + FP + 1e-8)
#     recall = TP / (TP + FN + 1e-8)
#     f1 = 2 * precision * recall / (precision + recall + 1e-8)

#     m = {
#         "TP": TP,
#         "FP": FP,
#         "FN": FN,
#         "precision": precision,
#         "recall": recall,
#         "f1": f1
#     }

#     page_metrics.append(m)
#     print(f"Page F1={m['f1']:.3f} | P={m['precision']:.3f}, R={m['recall']:.3f}")

# avg_f1 = np.mean([m["f1"] for m in page_metrics])
# print(f"\n🔍 Average F1-score across test pages: {avg_f1:.3f}")


Page F1=0.043 | P=0.025, R=0.141
Page F1=0.073 | P=0.048, R=0.153
Page F1=0.080 | P=0.054, R=0.153
Page F1=0.042 | P=0.024, R=0.149
Page F1=0.079 | P=0.052, R=0.167
Page F1=0.084 | P=0.056, R=0.175
Page F1=0.043 | P=0.024, R=0.174
Page F1=0.041 | P=0.025, R=0.112
Page F1=0.064 | P=0.041, R=0.146
Page F1=0.043 | P=0.025, R=0.142
Page F1=0.082 | P=0.054, R=0.170
Page F1=0.051 | P=0.031, R=0.141
Page F1=0.039 | P=0.026, R=0.085
Page F1=0.052 | P=0.034, R=0.110
Page F1=0.043 | P=0.026, R=0.136
Page F1=0.034 | P=0.020, R=0.115
Page F1=0.045 | P=0.030, R=0.091
Page F1=0.040 | P=0.025, R=0.096
Page F1=0.052 | P=0.033, R=0.122
Page F1=0.038 | P=0.023, R=0.112
Page F1=0.043 | P=0.025, R=0.146
Page F1=0.022 | P=0.012, R=0.099
Page F1=0.022 | P=0.012, R=0.088
Page F1=0.060 | P=0.041, R=0.107
Page F1=0.040 | P=0.023, R=0.147
Page F1=0.024 | P=0.014, R=0.081
Page F1=0.043 | P=0.026, R=0.127
Page F1=0.051 | P=0.034, R=0.103

🔍 Average F1-score across test pages: 0.049
